### Introduction
This is the main data analysis notebook, here I will clean the data and apply natural language processing techniques to extract the years of experience and the skills from the job descriptions. After which I will do a simple frequency count to see which among the two BI tools, Power BI or Tableau is most mentioned in job descricptions.

### Data wrangling

In [1]:
# Import all necessary libraries

import numpy as np
import pandas as pd
import re

import matplotlib.pyplot as plt
import seaborn as sns


I will start with cleaning the first dataset from glassdoor

In [2]:
#clean the first dataset from glassdoor

df1=pd.read_csv(r'C:\Users\Wambui\Desktop\Job_Market_Analysis-BI_Tools\glassdoor_data.csv')
df1.head()

,title,description,link
0,Tableau Developer,Job Title: Tableau Developer\nDescription: The...,https://www.glassdoor.com/job-listing/tableau-...
1,Power BI Co-Op,"About American Ordnance\nAt American Ordnance,...",https://www.glassdoor.com/job-listing/power-bi...
2,Power BI Designer,We are looking for a Microsoft PowerBI Designe...,https://www.glassdoor.com/job-listing/power-bi...
3,Power BI Developer,"19562\nIES Holdings\nSugar Land, Texas\n\nJob ...",https://www.glassdoor.com/job-listing/power-bi...
4,Lead Power BI Developer,We are seeking a Power BI Reporting Lead to dr...,https://www.glassdoor.com/job-listing/lead-pow...


In [3]:
#handle missingness, drop any row which missing values
df1.dropna(inplace=True)

For this analysis, I also want to examine if the preferrence for Power BI or Tableau is based on the specific data role, for example Data Analyst or Data Manager. We know the same job role, technically can be referred to in a myriad of ways, depending on the company. So the next step is to categorize the scrapped job titles into what I am calling 'core titles'. This way a 'Sales BI Analyst' is simply a 'BI Analyst', making analysis much easier.

In [4]:
#first check the variety of titles
df1['title'].value_counts()

title
Business Intelligence Analyst                             34
Power BI Developer                                        26
Data Analyst                                              21
Senior Power BI Developer                                 14
Tableau Developer                                         12
                                                          ..
Power BI Engineer (DAX, Power Query, Power BI Service)     1
Data Analyst - IT                                          1
EPIC Inpatient BI Developer                                1
Senior Business Intelligence Analyst                       1
Vice President, Analytics                                  1
Name: count, Length: 356, dtype: int64

In [5]:
title_keywords = {
    "data engineer": ["engineer", "engineering"],
    "data manager": ["manager", "management", "data director","president"],
    "data analyst": ["data analyst", "analytics", "analysis"],
    "bi analyst": ["bi analyst", "business intelligence analyst","business analyst","business intelligence","market intelligence"],
    "bi developer": ["bi developer", "business intelligence developer","developer","power bi"],
    "data scientist": ["scientist", "science","artificial intelligence"],
    "data architect": ["architect", "architecture"],
    "data specialist": ["specialist"],
    "data steward": ["steward", "librarian"],
    "data coordinator": ["coordinator", "data coordinator"],
    "data strategist": ["strategist", "strategy"],
    "data quality specialist": ["quality specialist", "quality assurance", "data quality"],
    "data educator":["lecturer","professor"],
    "data governance":["data governance"],
    "business development":["business development"]
    }


In [6]:
def get_core_title(title):
    title_lower = title.lower()

    # Check each key in the dictionary
    for core_title, keywords in title_keywords.items():
        # If any keyword matches, return the core job title
        if any(keyword in title_lower for keyword in keywords):
            return core_title
    
    return "unknown"  #if no match found

In [7]:
df1["core_title"] = df1["title"].apply(get_core_title)
df1[["title", "core_title"]]

,title,core_title
0,Tableau Developer,bi developer
1,Power BI Co-Op,bi developer
2,Power BI Designer,bi developer
3,Power BI Developer,bi developer
4,Lead Power BI Developer,bi developer
...,...,...
795,Data Analyst - IT,data analyst
796,Data Visualization Specialist,data specialist
797,EPIC Inpatient BI Developer,bi developer
798,Senior Business Intelligence Analyst,bi analyst


In [8]:
df1.core_title.value_counts()

core_title
bi developer        267
bi analyst          148
data analyst        124
unknown             104
data engineer        54
data manager         40
data specialist      33
data strategist      11
data architect        7
data coordinator      3
Name: count, dtype: int64

In [9]:
df1['source']='glassdoor'

In [10]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 791 entries, 0 to 799
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        791 non-null    object
 1   description  791 non-null    object
 2   link         791 non-null    object
 3   core_title   791 non-null    object
 4   source       791 non-null    object
dtypes: object(5)
memory usage: 37.1+ KB


Now I will clean the myjobmag dataset

In [11]:
df2=pd.read_csv(r'C:\Users\Wambui\Desktop\Job_Market_Analysis-BI_Tools\myjobmag_data.csv')
df2.head()

,title,description,link
0,Manager- Data Transformation Intelligence and ...,"Our ideal candidate is self-motivated, adaptab...",https://www.myjobmag.co.ke/job/manager-data-tr...
1,Manager – Data Transformation Intelligence and...,Job Summary\ninSupply Health Ltd. is seeking a...,https://www.myjobmag.co.ke/job/manager-data-tr...
2,Business Intelligence Analyst at Gertrude's Ch...,Key Responsibilities\nCollaborate with various...,https://www.myjobmag.co.ke/job/business-intell...
3,Senior Analyst: Business Intelligence at Cellu...,"ROLE PURPOSE:\nThe Senior Analyst, Business In...",https://www.myjobmag.co.ke/job/senior-analyst-...
4,Business Intelligence Analyst at SENRI Ltd. (A...,"As a Business Intelligence(BI) Analyst, you wi...",https://www.myjobmag.co.ke/job/business-intell...


In [12]:
#handle missingness, drop any row which missing values
df2.dropna(inplace=True)

Notice, for the job titles, they also include the company, we do not need this information, so I will right a regex function to remove all words after 'at' assuming that the words preceeding that are the actualy job titles. Some titles also have '-', the function will remove this also.

In [13]:
def remove_company_name(title):
    return re.sub(r'\s+(at|\||-)\s+.*', '', title, flags=re.IGNORECASE)

In [14]:
df2['cleaned_title']=df2['title'].apply(remove_company_name)
df2

,title,description,link,cleaned_title
0,Manager- Data Transformation Intelligence and ...,"Our ideal candidate is self-motivated, adaptab...",https://www.myjobmag.co.ke/job/manager-data-tr...,Manager- Data Transformation Intelligence and ...
1,Manager – Data Transformation Intelligence and...,Job Summary\ninSupply Health Ltd. is seeking a...,https://www.myjobmag.co.ke/job/manager-data-tr...,Manager – Data Transformation Intelligence and...
2,Business Intelligence Analyst at Gertrude's Ch...,Key Responsibilities\nCollaborate with various...,https://www.myjobmag.co.ke/job/business-intell...,Business Intelligence Analyst
3,Senior Analyst: Business Intelligence at Cellu...,"ROLE PURPOSE:\nThe Senior Analyst, Business In...",https://www.myjobmag.co.ke/job/senior-analyst-...,Senior Analyst: Business Intelligence
4,Business Intelligence Analyst at SENRI Ltd. (A...,"As a Business Intelligence(BI) Analyst, you wi...",https://www.myjobmag.co.ke/job/business-intell...,Business Intelligence Analyst
...,...,...,...,...
892,Business Development Internship at Adjacent Po...,Eligibility:\nInternship at APF is open to can...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Internship
893,Team Leader - Corporate Business Development (...,Key Responsibilities\nBusiness Growth & Strate...,https://www.myjobmag.co.ke/job/team-leader-cor...,Team Leader
894,Real Estate Business Partner CU CEA at Ericsson,What you will do:\nOwn and build strategic rel...,https://www.myjobmag.co.ke/job/real-estate-bus...,Real Estate Business Partner CU CEA
895,Relationship Officer- Business Development at ...,Job Summary:\nTo achieve business growth for t...,https://www.myjobmag.co.ke/job/relationship-of...,Relationship Officer- Business Development


In [15]:
df2["core_title"] = df2["cleaned_title"].apply(get_core_title)
df2[["cleaned_title", "core_title"]]

,cleaned_title,core_title
0,Manager- Data Transformation Intelligence and ...,data manager
1,Manager – Data Transformation Intelligence and...,data manager
2,Business Intelligence Analyst,bi analyst
3,Senior Analyst: Business Intelligence,bi analyst
4,Business Intelligence Analyst,bi analyst
...,...,...
892,Business Development Internship,business development
893,Team Leader,unknown
894,Real Estate Business Partner CU CEA,unknown
895,Relationship Officer- Business Development,business development


In [16]:
df2['core_title'].value_counts()

core_title
unknown                    253
data manager               192
business development       153
bi analyst                  72
data analyst                64
data engineer               56
data scientist              45
data educator               17
data specialist             17
bi developer                16
data coordinator             4
data governance              3
data quality specialist      2
data steward                 1
data strategist              1
data architect               1
Name: count, dtype: int64

In [17]:
df2.loc[df2['core_title']=='unknown'].tail(30)

,title,description,link,cleaned_title,core_title
769,Finance Business Partner at Cellulant Corporation,JOB DESCRIPTION:\nThe Finance Business Partner...,https://www.myjobmag.co.ke/job/finance-busines...,Finance Business Partner,unknown
773,Director of Business Operations at CDL Human R...,"Our client, a nonprofit organization is seekin...",https://www.myjobmag.co.ke/job/director-of-bus...,Director of Business Operations,unknown
776,Catering & Accommodation Trainer at James Flav...,"This is a full-time, on-site role located in T...",https://www.myjobmag.co.ke/job/catering-accomm...,Catering & Accommodation Trainer,unknown
780,Human Resource Business Partner at Solar Panda,The Human Resource Business Partner at Solar P...,https://www.myjobmag.co.ke/job/human-resource-...,Human Resource Business Partner,unknown
784,"Teacher of Business, Economics and Global Stud...",We are looking for candidates who:\nAre experi...,https://www.myjobmag.co.ke/job/teacher-of-busi...,"Teacher of Business, Economics and Global Studies",unknown
786,Human Resource Business Partner at Penda Health,About the Role: \nPenda Health is looking for ...,https://www.myjobmag.co.ke/job/human-resource-...,Human Resource Business Partner,unknown
787,Business Processing & Compliance Officer at Br...,Co-ordinate activities and support services wi...,https://www.myjobmag.co.ke/job/business-proces...,Business Processing & Compliance Officer,unknown
805,"Head, Business Channels Tribe at HF Group",Principle Accountabilities\nDevelop and execut...,https://www.myjobmag.co.ke/job/head-business-c...,"Head, Business Channels Tribe",unknown
806,Internship for Youth Aquaculture Graduates Und...,The internship program under the NORAD Grant a...,https://www.myjobmag.co.ke/job/internship-for-...,Internship for Youth Aquaculture Graduates Und...,unknown
818,"Senior Officer, Business Applications - 2 Post...","JOB PURPOSE\nThe Senior Officer, Business Appl...",https://www.myjobmag.co.ke/job/senior-officer-...,"Senior Officer, Business Applications",unknown


In [18]:
df2.loc[df2['core_title']=='business development'].tail(30)

,title,description,link,cleaned_title,core_title
807,Business Development Executive at SENRI Ltd. (...,Responsibilities:\nIdentify and develop new bu...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Executive,business development
809,Business Development Executive at JardineHR Co...,Our Client a Media Agency that specializes in ...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Executive,business development
812,Business Development Officer - Fixed Term at O...,JOB SUMMARY\nThe incumbent will be required to...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Officer,business development
813,Senior Business Development Officer at APA Lif...,KEY PRIMARY RESPONSIBILITIES\nSecure new healt...,https://www.myjobmag.co.ke/job/senior-business...,Senior Business Development Officer,business development
826,Business Development Officer at NCBA Group,Job Purpose Statement\nTo market and sell the ...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Officer,business development
830,Technology Business Development Executive at T...,What You’ll Do:\nIdentify & pursue new busines...,https://www.myjobmag.co.ke/job/technology-busi...,Technology Business Development Executive,business development
834,Business Development Officer – Alternative Cha...,Job Purpose Statement\nNCBA Insurance Company’...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Officer – Alternative Cha...,business development
835,Business Development Officer – Bancassurance a...,Job Purpose Statement\nNCBA Insurance Company’...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Officer – Bancassurance,business development
836,Business Development Officer at ABC EXPAT,We are seeking to ﬁll our Business Development...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Officer,business development
847,Business Development Officers - Insurance at M...,Key Responsibilities\nClient Acquisition - Ide...,https://www.myjobmag.co.ke/job/business-develo...,Business Development Officers,business development


From the looks of it, when I scrapped the MyJobMag Kenya website using the key word 'business intelligence analyst' it also picked roles that were related to business, such as 'business development'. Since these are not roles in the data field, I will drop them, together with any role categorized as 'unknown'.

In [19]:
df2_cleaned = df2.loc[~df2['core_title'].isin(['business development', 'unknown'])]


In [20]:
df2_cleaned['core_title'].value_counts()

core_title
data manager               192
bi analyst                  72
data analyst                64
data engineer               56
data scientist              45
data educator               17
data specialist             17
bi developer                16
data coordinator             4
data governance              3
data quality specialist      2
data steward                 1
data strategist              1
data architect               1
Name: count, dtype: int64

In [22]:
df2_cleaned['source']='myjobmag'
df2_cleaned.head(3)

C:\Users\Wambui\AppData\Local\Temp\ipykernel_22724\3101916060.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_cleaned['source']='myjobmag'


,title,description,link,cleaned_title,core_title,source
0,Manager- Data Transformation Intelligence and ...,"Our ideal candidate is self-motivated, adaptab...",https://www.myjobmag.co.ke/job/manager-data-tr...,Manager- Data Transformation Intelligence and ...,data manager,myjobmag
1,Manager – Data Transformation Intelligence and...,Job Summary\ninSupply Health Ltd. is seeking a...,https://www.myjobmag.co.ke/job/manager-data-tr...,Manager – Data Transformation Intelligence and...,data manager,myjobmag
2,Business Intelligence Analyst at Gertrude's Ch...,Key Responsibilities\nCollaborate with various...,https://www.myjobmag.co.ke/job/business-intell...,Business Intelligence Analyst,bi analyst,myjobmag


Now that I have 2 clean and categorized datasets, I can go ahead and merge them together.

In [23]:
df=pd.concat([df1,df2_cleaned],ignore_index=True)
df.head()

,title,description,link,core_title,source,cleaned_title
0,Tableau Developer,Job Title: Tableau Developer\nDescription: The...,https://www.glassdoor.com/job-listing/tableau-...,bi developer,glassdoor,NaN
1,Power BI Co-Op,"About American Ordnance\nAt American Ordnance,...",https://www.glassdoor.com/job-listing/power-bi...,bi developer,glassdoor,NaN
2,Power BI Designer,We are looking for a Microsoft PowerBI Designe...,https://www.glassdoor.com/job-listing/power-bi...,bi developer,glassdoor,NaN
3,Power BI Developer,"19562\nIES Holdings\nSugar Land, Texas\n\nJob ...",https://www.glassdoor.com/job-listing/power-bi...,bi developer,glassdoor,NaN
4,Lead Power BI Developer,We are seeking a Power BI Reporting Lead to dr...,https://www.glassdoor.com/job-listing/lead-pow...,bi developer,glassdoor,NaN


## Text normalization